# A2 — Lateral Inhibition Sparse Router (LI-SCR)

Wilson-Cowan iteratif inhibition + Hebbian güncelleme. Güçlü tokenlar zayıfları bastırır, sparse aktivasyon üretir.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/A2_LI_SCR_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "A2_LI_SCR",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  100,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (LI-SCR)
    "li_lambda":         0.5,        # inhibition şiddeti
    "li_num_iters":      3,          # W-C iterasyon
    "li_hebbian_lr":     0.01,       # inhibition matrisi update hızı
    "li_weight_decay":   0.001,      # W matrisi decay
    "li_min_activation": 0.01,       # WTA collapse önleme
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Lateral Inhibition Sparse Router ────────────────
class LateralInhibitionRouter(BaseRouterAttention):
    def __init__(self, gpt2_config, method_params):
        super().__init__(gpt2_config, method_params)
        mp = method_params
        self.lam = float(mp.get("li_lambda", 0.5))
        self.num_iters = int(mp.get("li_num_iters", 3))
        self.hebbian_lr = float(mp.get("li_hebbian_lr", 0.01))
        self.weight_decay_rate = float(mp.get("li_weight_decay", 0.001))
        self.min_activation = float(mp.get("li_min_activation", 0.01))
        max_pos = gpt2_config.n_positions
        self.W_inhib = nn.Parameter(torch.randn(max_pos, max_pos) * 0.01)

    def _inhibit(self, scores, seq_len):
        W = self.W_inhib[:seq_len, :seq_len]
        alpha = scores
        for _ in range(self.num_iters):
            inhib = self.lam * torch.matmul(alpha, W.T)
            alpha = torch.relu(scores - inhib)
            alpha = alpha.clamp(min=self.min_activation)
        return alpha

    def _hebbian_update(self, alpha, seq_len):
        if not self.training:
            return
        with torch.no_grad():
            mean_alpha = alpha.mean(dim=(0, 1))
            corr = mean_alpha.T @ mean_alpha
            dW = self.hebbian_lr * corr - self.weight_decay_rate * self.W_inhib[:seq_len, :seq_len]
            self.W_inhib.data[:seq_len, :seq_len] += dW

    def _attend(self, q, k, v, hidden_states, attention_mask):
        raw = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        causal = self._causal_mask(q.size(-2), k.size(-2), q.device)
        raw = raw.masked_fill(~causal, 0.0)
        alpha = self._inhibit(raw, raw.size(-1))
        alpha = alpha.masked_fill(~causal, 0.0)
        alpha = alpha / (alpha.sum(-1, keepdim=True) + 1e-9)
        self._hebbian_update(alpha.detach(), alpha.size(-1))
        alpha_drop = self.attn_dropout(alpha)
        out = torch.matmul(alpha_drop, v)
        self._last_stats = {"sparsity": float((alpha < self.min_activation * 1.01).float().mean().item())}
        return out, alpha

print("LateralInhibitionRouter tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    wrapper = GPT2Wrapper(params["model_name"])
    wrapper.inject_attention(LateralInhibitionRouter, params)
    print(f"LI-SCR injected → {sum(p.numel() for p in wrapper.parameters()):,} params")
    return wrapper, None

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL: {result['final_metrics']['test_ppl']:.4f}")
print(f"Final test BPC: {result['final_metrics']['test_bpc']:.4f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")